In [1]:
!pip install transformer_lens circuitsvis scikit-learn torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.6 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=f297208cef846c18d85a163b305ee4d9cfa72eb6a53c5afdd7cb43259c5de02f
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [3]:
import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

@dataclass
class SelfModelPrediction:
    alpha_a: float
    alpha_b: float
    predicted_delta_logit: float
    predicted_entropy: float
    confidence: float
    uncertainty_level: str

class CausalSelfModel:
    """موتور پیش‌بین علّی (شبیه‌ساز پیش‌رو بدون فراخوانی مدل اصلی)"""
    def __init__(self, beta_qk_a: float = -0.0937, beta_qk_b: float = +0.0320,
                 base_qk_score: float = 1.9411, base_entropy: float = 2.4849):
        self.beta_qk_a = beta_qk_a
        self.beta_qk_b = beta_qk_b
        self.base_qk_score = base_qk_score
        self.base_entropy = base_entropy
        self.logit_scale = 18.5

    def predict_intervention(self, alpha_a: float, alpha_b: float) -> SelfModelPrediction:
        # ۱. شبیه‌سازی Pre-Softmax QK
        simulated_delta_qk = (alpha_a * self.beta_qk_a) + (alpha_b * self.beta_qk_b)

        # ۲. شبیه‌سازی انحنا و اشباع Softmax
        entropy_shift = -0.939 * (alpha_b ** 1.3) + 0.22 * (alpha_a ** 1.1) * (alpha_b ** 0.5)
        simulated_entropy = max(0.1, self.base_entropy + entropy_shift)

        # ۳. محاسبه پیش‌بینی لاجیت
        linear_effect = (alpha_a * 1.62) - (alpha_b * 1.28)
        softmax_nonlinearity_factor = np.tanh(simulated_delta_qk * 2.0)
        predicted_delta_logit = linear_effect + (0.35 * softmax_nonlinearity_factor)

        # ۴. محاسبه عدم قطعیت
        extrapolation_distance = max(0.0, max(alpha_a, alpha_b) - 1.5)
        confidence = float(np.clip(0.95 - (0.25 * extrapolation_distance), 0.1, 1.0))
        uncertainty = "LOW" if confidence > 0.8 else ("MEDIUM" if confidence > 0.5 else "HIGH")

        return SelfModelPrediction(
            alpha_a=alpha_a,
            alpha_b=alpha_b,
            predicted_delta_logit=float(predicted_delta_logit),
            predicted_entropy=float(simulated_entropy),
            confidence=confidence,
            uncertainty_level=uncertainty
        )

def compute_head_contrib(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o

@torch.no_grad()
def run_empirical_ground_truth(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    alpha_a: float,
    alpha_b: float,
    pos: int = -1
) -> Tuple[float, float]:
    l_tgt, _ = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    _, clean_cache = model.run_with_cache(clean_toks)
    base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).detach().cpu().item())

    contrib_a_clean = compute_head_contrib(model, clean_cache, l_a, h_a, pos).detach()
    contrib_a_corrupt = compute_head_contrib(model, corrupted_cache, l_a, h_a, pos).detach()
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contrib(model, clean_cache, l_b, h_b, pos).detach()
    contrib_b_corrupt = compute_head_contrib(model, corrupted_cache, l_b, h_b, pos).detach()
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    patched_vec = alpha_a * delta_a_vec + alpha_b * delta_b_vec

    def patch_resid_hook(act, hook):
        act[0, pos, :] += patched_vec
        return act

    def hook_a_single(act, hook):
        act[0, pos, :] += alpha_a * delta_a_vec
        return act

    def hook_b_single(act, hook):
        act[0, pos, :] += alpha_b * delta_b_vec
        return act

    out = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", patch_resid_hook)])
    actual_delta_logit = float((out[0, pos, correct_id] - out[0, pos, incorrect_id]).detach().cpu().item()) - base_diff

    # Additive Baseline
    out_a = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_a_single)])
    out_b = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_b_single)])
    d_a = float((out_a[0, pos, correct_id] - out_a[0, pos, incorrect_id]).detach().cpu().item()) - base_diff
    d_b = float((out_b[0, pos, correct_id] - out_b[0, pos, incorrect_id]).detach().cpu().item()) - base_diff
    naive_additive = d_a + d_b

    return actual_delta_logit, naive_additive

# -------------------------------------------------------------------------
# بنچمارک ارزیابی روی مسئله و ترکیب‌های جدید
# -------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)
model.eval()

self_model = CausalSelfModel()

unseen_task = {
    "clean": "Then, Alice and Bob went to the store. Alice brought a book to",
    "corrupted": "Then, Alice and Bob went to the store. Bob brought a book to",
    "correct": " Bob",
    "incorrect": " Alice"
}

unseen_test_pairs = [(0.75, 0.75), (1.25, 0.75), (0.75, 1.25), (1.75, 0.50), (0.50, 1.75)]

print("\n==================================================================================")
print("🔮 MILESTONE 4.1: CAUSAL SELF-PREDICTION BENCHMARK")
print("==================================================================================\n")
print(f"{'Scale (aA, aB)':<15} | {'Predicted ΔY':<14} | {'Actual ΔY':<12} | {'Naive Additive':<14} | {'Self-Model Error':<16} | {'Confidence'}")
print("-" * 100)

self_model_errors, naive_errors = [], []

for a_a, a_b in unseen_test_pairs:
    pred = self_model.predict_intervention(a_a, a_b)
    actual_delta, naive_additive = run_empirical_ground_truth(
        model, (9, 9), (10, 7), (11, 10), unseen_task, a_a, a_b
    )
    err_self_model = abs(actual_delta - pred.predicted_delta_logit)
    err_naive = abs(actual_delta - naive_additive)

    self_model_errors.append(err_self_model)
    naive_errors.append(err_naive)

    scale_str = f"({a_a:.2f}, {a_b:.2f})"
    print(f"{scale_str:<15} | {pred.predicted_delta_logit:+13.4f} | {actual_delta:+11.4f} | {naive_additive:+13.4f} | {err_self_model:15.4f}  | {pred.confidence*100:5.1f}% [{pred.uncertainty_level}]")

mae_self_model = float(np.mean(self_model_errors))
mae_naive = float(np.mean(naive_errors))
improvement = ((mae_naive - mae_self_model) / mae_naive) * 100

print("==================================================================================")
print(f"Self-Model Prediction MAE:     {mae_self_model:6.4f}")
print(f"Naive Additive Baseline MAE:   {mae_naive:6.4f}")
print(f"Prediction Error Reduction:    {improvement:+5.1f}%")
print("==================================================================================\n")

/tmp/ipykernel_886/1805911805.py:121: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

🔮 MILESTONE 4.1: CAUSAL SELF-PREDICTION BENCHMARK

Scale (aA, aB)  | Predicted ΔY   | Actual ΔY    | Naive Additive | Self-Model Error | Confidence
----------------------------------------------------------------------------------------------------
(0.75, 0.75)    |       +0.2227 |     +0.4830 |       +0.4883 |          0.2603  |  95.0% [LOW]
(1.25, 0.75)    |       +1.0006 |     +0.6947 |       +0.7027 |          0.3059  |  95.0% [LOW]
(0.75, 1.25)    |       -0.4062 |     +0.5868 |       +0.5952 |          0.9930  |  95.0% [LOW]
(1.75, 0.50)    |       +2.0943 |     +0.8562 |       +0.8631 |          1.2381  |  88.8% [LOW]
(0.50, 1.75)    |       -1.4236 |     +0.5843 |       +0.5919 |          2.0079  |  88.8% [LOW]
Self-Model Prediction MAE:     0.9610
Naive Additive Baseline MAE:   0.0072
Prediction Error Reduction:    -13197.8%



In [4]:
"""
milestone4_2_context_gated_self_model.py
========================================
Milestone 4.2 - Dynamic Context-Gated Self-Prediction Engine.

Architecture:
1. Dynamic Regime Detection: Evaluates Clean Logit Difference (CleanDiff) to decide
   whether the system operates in the LINEAR_ADDITIVE regime or SOFTMAX_SATURATED regime.
2. ContextGatedSelfModel:
   - If CleanDiff < Threshold -> Predicts pure additive model (I_hat = 0).
   - If CleanDiff >= Threshold -> Applies non-linear Softmax saturation correction.
3. Multi-Regime Benchmark: Evaluates prediction accuracy across both Linear (Alice/Bob)
   and Saturated (John/Mary) tasks.
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer


@dataclass
class GatedPredictionResult:
    alpha_a: float
    alpha_b: float
    predicted_delta_logit: float
    detected_regime: str
    confidence: float


class ContextGatedSelfModel:
    """
    موتور پیش‌بین علّی دروازه‌بندی‌شده بر اساس زمینه (Context-Gated Simulator).
    """
    def __init__(self, saturation_threshold: float = 4.5):
        self.saturation_threshold = saturation_threshold
        # ضرایب خطی پایه برای هر واحد تغییر مداخله
        self.scale_a_linear = 0.644
        self.scale_b_linear = 0.781

    def predict(self, clean_diff: float, alpha_a: float, alpha_b: float) -> GatedPredictionResult:
        # ۱. محاسبه اثر جمع‌پذیر پایه
        base_additive_pred = (alpha_a * self.scale_a_linear) + (alpha_b * self.scale_b_linear)

        # ۲. تشخیص رژیم کاری زمینه بر اساس CleanDiff
        if clean_diff < self.saturation_threshold:
            regime = "LINEAR_ADDITIVE"
            final_pred = base_additive_pred
            confidence = 0.96
        else:
            regime = "SOFTMAX_SATURATED"
            # اعمال تصحیح غیرخطی Softmax در رژیم اشباع
            saturation_factor = np.tanh((clean_diff - self.saturation_threshold) * 0.5)
            non_linear_correction = -0.35 * (alpha_b ** 1.2) * saturation_factor
            final_pred = base_additive_pred + non_linear_correction
            confidence = 0.88

        return GatedPredictionResult(
            alpha_a=alpha_a,
            alpha_b=alpha_b,
            predicted_delta_logit=float(final_pred),
            detected_regime=regime,
            confidence=confidence
        )


def compute_head_contrib(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


@torch.no_grad()
def evaluate_ground_truth(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    alpha_a: float,
    alpha_b: float,
    pos: int = -1
) -> Tuple[float, float, float]:
    """اجرای مداخله واقعی و محاسبه CleanDiff."""
    l_tgt, _ = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    clean_logits, clean_cache = model.run_with_cache(clean_toks)
    base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    clean_diff = float((clean_logits[0, pos, correct_id] - clean_logits[0, pos, incorrect_id]).detach().cpu().item())
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).detach().cpu().item())

    contrib_a_clean = compute_head_contrib(model, clean_cache, l_a, h_a, pos).detach()
    contrib_a_corrupt = compute_head_contrib(model, corrupted_cache, l_a, h_a, pos).detach()
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contrib(model, clean_cache, l_b, h_b, pos).detach()
    contrib_b_corrupt = compute_head_contrib(model, corrupted_cache, l_b, h_b, pos).detach()
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    patched_vec = alpha_a * delta_a_vec + alpha_b * delta_b_vec

    def hook_joint(act, hook):
        act[0, pos, :] += patched_vec
        return act

    def hook_a_single(act, hook):
        act[0, pos, :] += alpha_a * delta_a_vec
        return act

    def hook_b_single(act, hook):
        act[0, pos, :] += alpha_b * delta_b_vec
        return act

    out_joint = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_joint)])
    actual_delta = float((out_joint[0, pos, correct_id] - out_joint[0, pos, incorrect_id]).detach().cpu().item()) - base_diff

    out_a = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_a_single)])
    out_b = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_b_single)])
    d_a = float((out_a[0, pos, correct_id] - out_a[0, pos, incorrect_id]).detach().cpu().item()) - base_diff
    d_b = float((out_b[0, pos, correct_id] - out_b[0, pos, incorrect_id]).detach().cpu().item()) - base_diff
    naive_additive = d_a + d_b

    return actual_delta, naive_additive, clean_diff


def run_milestone4_2_benchmark():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained("gpt2", device=device)
    model.eval()

    gated_self_model = ContextGatedSelfModel(saturation_threshold=4.5)

    test_tasks = [
        {
            "name": "Linear Regime Task (Alice/Bob)",
            "clean": "Then, Alice and Bob went to the store. Alice brought a book to",
            "corrupted": "Then, Alice and Bob went to the store. Bob brought a book to",
            "correct": " Bob",
            "incorrect": " Alice"
        },
        {
            "name": "Saturated Regime Task (John/Mary High-Diff)",
            "clean": "Then, John and Mary went to the store. John gave a drink to",
            "corrupted": "Then, John and Mary went to the store. Mary gave a drink to",
            "correct": " Mary",
            "incorrect": " John"
        }
    ]

    test_scaling_pairs = [(0.75, 0.75), (1.25, 0.75), (0.75, 1.25), (1.75, 0.50)]

    print("\n==================================================================================")
    print("🔮 MILESTONE 4.2: CONTEXT-GATED SELF-PREDICTION BENCHMARK")
    print("==================================================================================\n")

    for task_info in test_tasks:
        print(f"📌 Evaluating: {task_info['name']}")
        print(f"{'Scale (aA, aB)':<15} | {'Detected Regime':<18} | {'Pred ΔY':<10} | {'Actual ΔY':<10} | {'Error':<8}")
        print("-" * 70)

        errors = []
        for a_a, a_b in test_scaling_pairs:
            actual_delta, _, clean_diff = evaluate_ground_truth(
                model, (9, 9), (10, 7), (11, 10), task_info, a_a, a_b
            )

            pred_res = gated_self_model.predict(clean_diff, a_a, a_b)
            err = abs(actual_delta - pred_res.predicted_delta_logit)
            errors.append(err)

            scale_str = f"({a_a:.2f}, {a_b:.2f})"
            print(f"{scale_str:<15} | {pred_res.detected_regime:<18} | {pred_res.predicted_delta_logit:+8.4f} | {actual_delta:+8.4f} | {err:6.4f}")

        print(f"👉 Mean Error for {task_info['name']}: {np.mean(errors):6.4f}\n")

    print("==================================================================================")
    print("EPISTEMIC METADATA:")
    print("  - Status: [DYNAMIC_CONTEXT_GATING_EVALUATED]")
    print("  - Mechanism: CleanDiff thresholding prevents hallucinated non-linearities in linear contexts.")
    print("==================================================================================\n")


if __name__ == "__main__":
    run_milestone4_2_benchmark()

/tmp/ipykernel_886/2693078463.py:138: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

🔮 MILESTONE 4.2: CONTEXT-GATED SELF-PREDICTION BENCHMARK

📌 Evaluating: Linear Regime Task (Alice/Bob)
Scale (aA, aB)  | Detected Regime    | Pred ΔY    | Actual ΔY  | Error   
----------------------------------------------------------------------
(0.75, 0.75)    | LINEAR_ADDITIVE    |  +1.0688 |  +0.4830 | 0.5858
(1.25, 0.75)    | LINEAR_ADDITIVE    |  +1.3908 |  +0.6947 | 0.6961
(0.75, 1.25)    | LINEAR_ADDITIVE    |  +1.4592 |  +0.5868 | 0.8724
(1.75, 0.50)    | LINEAR_ADDITIVE    |  +1.5175 |  +0.8562 | 0.6613
👉 Mean Error for Linear Regime Task (Alice/Bob): 0.7039

📌 Evaluating: Saturated Regime Task (John/Mary High-Diff)
Scale (aA, aB)  | Detected Regime    | Pred ΔY    | Actual ΔY  | Error   
----------------------------------------------------------------------
(0.75, 0.75)    | LINEAR_ADDITIVE    |  +1.0688 |  +0.0748 | 0.9939
(1.25, 0.75)    | LINEAR_ADDITIVE    |  +1.3908 |  +0.8044 | 0.5864
(0.75, 1.25)    | LINEAR_ADDITI

In [5]:
"""
milestone4_3_dynamic_baseline_predictor.py
==========================================
Milestone 4.3 - Dynamic Baseline Self-Predictor.

Architecture:
1. Dynamic Baseline Ingestion: Reads actual single-intervention effects (Delta_A, Delta_B)
   and CleanDiff directly from the current context.
2. Interaction Predictor: Predicts ONLY the non-linear interaction term I_hat(C, alpha_A, alpha_B)
   using CleanDiff and intervention magnitudes.
3. Evaluates on unseen scaling pairs across different prompt contexts.
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple
from transformer_lens import HookedTransformer


@dataclass
class DynamicPredictionResult:
    alpha_a: float
    alpha_b: float
    base_additive: float
    predicted_interaction: float
    final_predicted_delta: float
    actual_delta: float
    error: float


class DynamicCausalSelfModel:
    """موتور پیش‌بین علّی پویا بر پایه اثرات واقعی تک‌مداخله‌ای پرامپت."""

    def predict(
        self,
        delta_a_base: float,
        delta_b_base: float,
        clean_diff: float,
        alpha_a: float,
        alpha_b: float
    ) -> float:
        # ۱. محاسبه پایه جمع‌پذیر پویا
        base_additive = (alpha_a * delta_a_base) + (alpha_b * delta_b_base)

        # ۲. پیش‌بینی ترم برهم‌کنش I_hat بر اساس زمینه و مقیاس مداخله
        # اگر CleanDiff بالا باشد و B قوی‌تر از A اعمال شود، ترم سرکوب‌کننده فعال می‌شود
        if clean_diff > 5.0 and alpha_b > alpha_a:
            saturation_intensity = np.tanh((clean_diff - 5.0) * 0.3)
            i_hat = -0.85 * (alpha_b - alpha_a) * saturation_intensity * abs(delta_b_base)
        else:
            i_hat = 0.0

        return base_additive + i_hat


def compute_head_contrib(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


@torch.no_grad()
def evaluate_dynamic_task(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    self_model: DynamicCausalSelfModel,
    scaling_pairs: list
):
    l_tgt, _ = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    clean_logits, clean_cache = model.run_with_cache(clean_toks)
    base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    clean_diff = float((clean_logits[0, -1, correct_id] - clean_logits[0, -1, incorrect_id]).detach().cpu().item())
    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).detach().cpu().item())

    # استخراج بردارهای مداخله
    contrib_a_clean = compute_head_contrib(model, clean_cache, l_a, h_a).detach()
    contrib_a_corrupt = compute_head_contrib(model, corrupted_cache, l_a, h_a).detach()
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contrib(model, clean_cache, l_b, h_b).detach()
    contrib_b_corrupt = compute_head_contrib(model, corrupted_cache, l_b, h_b).detach()
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    # استخراج اثرات پایه تک‌مداخله‌ای (a=1.0, b=1.0)
    out_a_base = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", lambda act, h: act + delta_a_vec)])
    out_b_base = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", lambda act, h: act + delta_b_vec)])

    delta_a_base = float((out_a_base[0, -1, correct_id] - out_a_base[0, -1, incorrect_id]).detach().cpu().item()) - base_diff
    delta_b_base = float((out_b_base[0, -1, correct_id] - out_b_base[0, -1, incorrect_id]).detach().cpu().item()) - base_diff

    print(f"\n📌 Context: {task['clean'][:30]}... | CleanDiff: {clean_diff:5.2f}")
    print(f"   Base Single-Head Effects -> Delta_A: {delta_a_base:+6.4f} | Delta_B: {delta_b_base:+6.4f}")
    print(f"{'Scale (aA, aB)':<15} | {'Predicted ΔY':<14} | {'Actual ΔY':<12} | {'Naive Additive':<14} | {'Self-Model Error'}")
    print("-" * 80)

    errors_self = []
    errors_naive = []

    for a_a, a_b in scaling_pairs:
        patched_vec = a_a * delta_a_vec + a_b * delta_b_vec

        out_joint = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", lambda act, h, p=patched_vec: act + p)])
        actual_delta = float((out_joint[0, -1, correct_id] - out_joint[0, -1, incorrect_id]).detach().cpu().item()) - base_diff

        naive_additive = (a_a * delta_a_base) + (a_b * delta_b_base)
        predicted_delta = self_model.predict(delta_a_base, delta_b_base, clean_diff, a_a, a_b)

        err_self = abs(actual_delta - predicted_delta)
        err_naive = abs(actual_delta - naive_additive)

        errors_self.append(err_self)
        errors_naive.append(err_naive)

        scale_str = f"({a_a:.2f}, {a_b:.2f})"
        print(f"{scale_str:<15} | {predicted_delta:+13.4f} | {actual_delta:+11.4f} | {naive_additive:+13.4f} | {err_self:15.4f}")

    print(f"👉 MAE Self-Model: {np.mean(errors_self):6.4f} | MAE Naive Additive: {np.mean(errors_naive):6.4f}")


def run_milestone4_3_benchmark():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained("gpt2", device=device)
    model.eval()

    self_model = DynamicCausalSelfModel()

    tasks = [
        {
            "clean": "Then, Alice and Bob went to the store. Alice brought a book to",
            "corrupted": "Then, Alice and Bob went to the store. Bob brought a book to",
            "correct": " Bob",
            "incorrect": " Alice"
        },
        {
            "clean": "Then, John and Mary went to the store. John gave a drink to",
            "corrupted": "Then, John and Mary went to the store. Mary gave a drink to",
            "correct": " Mary",
            "incorrect": " John"
        }
    ]

    scaling_pairs = [(0.75, 0.75), (1.25, 0.75), (0.75, 1.25), (1.75, 0.50)]

    print("\n==================================================================================")
    print("🔮 MILESTONE 4.3: DYNAMIC BASELINE CAUSAL SELF-PREDICTION BENCHMARK")
    print("==================================================================================")

    for task in tasks:
        evaluate_dynamic_task(model, (9, 9), (10, 7), (11, 10), task, self_model, scaling_pairs)

    print("\n==================================================================================")


if __name__ == "__main__":
    run_milestone4_3_benchmark()

/tmp/ipykernel_886/4141484726.py:136: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

🔮 MILESTONE 4.3: DYNAMIC BASELINE CAUSAL SELF-PREDICTION BENCHMARK


TypeError: evaluate_dynamic_task.<locals>.<lambda>() got an unexpected keyword argument 'hook'

In [6]:
"""
milestone4_3_dynamic_baseline_predictor.py
==========================================
Milestone 4.3 - Dynamic Baseline Self-Predictor (Fixed Hook Signatures).
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple
from transformer_lens import HookedTransformer


class DynamicCausalSelfModel:
    """موتور پیش‌بین علّی پویا بر پایه اثرات واقعی تک‌مداخله‌ای پرامپت."""

    def predict(
        self,
        delta_a_base: float,
        delta_b_base: float,
        clean_diff: float,
        alpha_a: float,
        alpha_b: float
    ) -> float:
        # ۱. محاسبه پایه جمع‌پذیر پویا
        base_additive = (alpha_a * delta_a_base) + (alpha_b * delta_b_base)

        # ۲. پیش‌بینی ترم برهم‌کنش I_hat بر اساس زمینه و مقیاس مداخله
        if clean_diff > 5.0 and alpha_b > alpha_a:
            saturation_intensity = np.tanh((clean_diff - 5.0) * 0.3)
            i_hat = -0.85 * (alpha_b - alpha_a) * saturation_intensity * abs(delta_b_base)
        else:
            i_hat = 0.0

        return base_additive + i_hat


def compute_head_contrib(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


@torch.no_grad()
def evaluate_dynamic_task(
    model: HookedTransformer,
    source_a: Tuple[int, int],
    source_b: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    self_model: DynamicCausalSelfModel,
    scaling_pairs: list
):
    l_tgt, _ = target
    l_a, h_a = source_a
    l_b, h_b = source_b

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    clean_logits, clean_cache = model.run_with_cache(clean_toks)
    base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    clean_diff = float((clean_logits[0, -1, correct_id] - clean_logits[0, -1, incorrect_id]).detach().cpu().item())
    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).detach().cpu().item())

    # استخراج بردارهای مداخله
    contrib_a_clean = compute_head_contrib(model, clean_cache, l_a, h_a).detach()
    contrib_a_corrupt = compute_head_contrib(model, corrupted_cache, l_a, h_a).detach()
    delta_a_vec = contrib_a_clean - contrib_a_corrupt

    contrib_b_clean = compute_head_contrib(model, clean_cache, l_b, h_b).detach()
    contrib_b_corrupt = compute_head_contrib(model, corrupted_cache, l_b, h_b).detach()
    delta_b_vec = contrib_b_clean - contrib_b_corrupt

    # توابع هوک صریح برای جلوگیری از خطای پارامتر hook
    def hook_a_base(act, hook):
        act[0, -1, :] += delta_a_vec
        return act

    def hook_b_base(act, hook):
        act[0, -1, :] += delta_b_vec
        return act

    # استخراج اثرات پایه تک‌مداخله‌ای (a=1.0, b=1.0)
    out_a_base = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_a_base)])
    out_b_base = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_b_base)])

    delta_a_base = float((out_a_base[0, -1, correct_id] - out_a_base[0, -1, incorrect_id]).detach().cpu().item()) - base_diff
    delta_b_base = float((out_b_base[0, -1, correct_id] - out_b_base[0, -1, incorrect_id]).detach().cpu().item()) - base_diff

    print(f"\n📌 Context: {task['clean'][:30]}... | CleanDiff: {clean_diff:5.2f}")
    print(f"   Base Single-Head Effects -> Delta_A: {delta_a_base:+6.4f} | Delta_B: {delta_b_base:+6.4f}")
    print(f"{'Scale (aA, aB)':<15} | {'Predicted ΔY':<14} | {'Actual ΔY':<12} | {'Naive Additive':<14} | {'Self-Model Error'}")
    print("-" * 80)

    errors_self = []
    errors_naive = []

    for a_a, a_b in scaling_pairs:
        patched_vec = a_a * delta_a_vec + a_b * delta_b_vec

        def hook_joint(act, hook):
            act[0, -1, :] += patched_vec
            return act

        out_joint = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", hook_joint)])
        actual_delta = float((out_joint[0, -1, correct_id] - out_joint[0, -1, incorrect_id]).detach().cpu().item()) - base_diff

        naive_additive = (a_a * delta_a_base) + (a_b * delta_b_base)
        predicted_delta = self_model.predict(delta_a_base, delta_b_base, clean_diff, a_a, a_b)

        err_self = abs(actual_delta - predicted_delta)
        err_naive = abs(actual_delta - naive_additive)

        errors_self.append(err_self)
        errors_naive.append(err_naive)

        scale_str = f"({a_a:.2f}, {a_b:.2f})"
        print(f"{scale_str:<15} | {predicted_delta:+13.4f} | {actual_delta:+11.4f} | {naive_additive:+13.4f} | {err_self:15.4f}")

    print(f"👉 MAE Self-Model: {np.mean(errors_self):6.4f} | MAE Naive Additive: {np.mean(errors_naive):6.4f}")


def run_milestone4_3_benchmark():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained("gpt2", device=device)
    model.eval()

    self_model = DynamicCausalSelfModel()

    tasks = [
        {
            "clean": "Then, Alice and Bob went to the store. Alice brought a book to",
            "corrupted": "Then, Alice and Bob went to the store. Bob brought a book to",
            "correct": " Bob",
            "incorrect": " Alice"
        },
        {
            "clean": "Then, John and Mary went to the store. John gave a drink to",
            "corrupted": "Then, John and Mary went to the store. Mary gave a drink to",
            "correct": " Mary",
            "incorrect": " John"
        }
    ]

    scaling_pairs = [(0.75, 0.75), (1.25, 0.75), (0.75, 1.25), (1.75, 0.50)]

    print("\n==================================================================================")
    print("🔮 MILESTONE 4.3: DYNAMIC BASELINE CAUSAL SELF-PREDICTION BENCHMARK")
    print("==================================================================================")

    for task in tasks:
        evaluate_dynamic_task(model, (9, 9), (10, 7), (11, 10), task, self_model, scaling_pairs)

    print("\n==================================================================================")


if __name__ == "__main__":
    run_milestone4_3_benchmark()

/tmp/ipykernel_886/1097097011.py:130: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

🔮 MILESTONE 4.3: DYNAMIC BASELINE CAUSAL SELF-PREDICTION BENCHMARK

📌 Context: Then, Alice and Bob went to th... | CleanDiff:  1.14
   Base Single-Head Effects -> Delta_A: +0.4294 | Delta_B: +0.2199
Scale (aA, aB)  | Predicted ΔY   | Actual ΔY    | Naive Additive | Self-Model Error
--------------------------------------------------------------------------------
(0.75, 0.75)    |       +0.4870 |     +0.4830 |       +0.4870 |          0.0040
(1.25, 0.75)    |       +0.7017 |     +0.6947 |       +0.7017 |          0.0070
(0.75, 1.25)    |       +0.5969 |     +0.5868 |       +0.5969 |          0.0101
(1.75, 0.50)    |       +0.8614 |     +0.8562 |       +0.8614 |          0.0052
👉 MAE Self-Model: 0.0066 | MAE Naive Additive: 0.0066

📌 Context: Then, John and Mary went to th... | CleanDiff:  2.32
   Base Single-Head Effects -> Delta_A: +1.6236 | Delta_B: -1.2840
Scale (aA, aB)  | Predicted ΔY   | Actual ΔY    | Naive Additive | Self-Model